# Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import ExcelWriter

import torch
import torch.nn as nn
from torch import Tensor
from torch.autograd import Variable
import torch.nn.functional as F
from torchsummary import summary
from torch.backends import cudnn
cudnn.benchmark = False
cudnn.deterministic = True

gpus = [0]
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import sklearn
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import cohen_kappa_score


from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce

import datetime
import time
import warnings
warnings.filterwarnings("ignore")
import scipy
import random
import math
from IPython.display import display

import pennylane as qml
from pennylane_qiskit import AerDevice

# GPU

In [ ]:
if torch.cuda.is_available():
    print("CUDA is available")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operating in device: {device}")
print(torch.__version__)
torch.cuda.empty_cache()

# Model

In [ ]:
class CircuitBuilder:
    def __init__(self, n_qubits, n_qlayers):
        self.n_qubits = n_qubits
        self.n_qlayers = n_qlayers
        self.input_dim = 2 ** n_qubits

    def __call__(self, inputs, weights):
        # AmplitudeEncoding with automatic padding
        qml.AmplitudeEmbedding(inputs, wires=range(self.n_qubits), pad_with=None, normalize=True)

        for layer in range(self.n_qlayers):
            for q in range(self.n_qubits):
                qml.RZ(weights[layer * 2, q, 1], wires=q)
                qml.RY(weights[layer * 2, q, 0], wires=q)
                qml.RZ(weights[layer * 2, q, 2], wires=q)

            for q in range(self.n_qubits):
                qml.CNOT(wires=[q, (q + 1) % self.n_qubits])

            for q in range(self.n_qubits):
                qml.RZ(weights[layer * 2 + 1, q, 1], wires=q)
                qml.RY(weights[layer * 2 + 1, q, 0], wires=q)
                qml.RZ(weights[layer * 2 + 1, q, 2], wires=q)

            qml.CNOT(wires=[3, 0])
            qml.CNOT(wires=[3, 2])
            qml.CNOT(wires=[2, 1])
            qml.CNOT(wires=[1, 0])

        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]


class MyQuantumCircuit(nn.Module):
    def __init__(self, n_qubits=4, n_qlayers=1):
        super().__init__()

        self.n_qubits = n_qubits
        self.n_qlayers = n_qlayers
        self.dev = qml.device("default.qubit", wires=self.n_qubits)

        self.circuit = CircuitBuilder(self.n_qubits, self.n_qlayers)
        self.qnode = qml.QNode(self.circuit, self.dev, interface="torch")
        self.qlayer = qml.qnn.TorchLayer(
            self.qnode,
            weight_shapes=self._get_weight_shapes()
        )

    def _get_weight_shapes(self):
        return {"weights": (self.n_qlayers * 2, self.n_qubits, 3)}

    def forward(self, x: Tensor) -> Tensor:
        return self.qlayer(x)



In [ ]:
class MEEGNeX(nn.Module):
    def __init__(self,n_channels=64,
                 dropout=0.5):
        super().__init__()

        # ---------- Block 1 ----------
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 8,kernel_size=(1, 64), padding='same',bias=False),
            nn.BatchNorm2d(8),
            nn.ELU()
        )
        # Depthwise spatial convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(8,16,kernel_size=(n_channels, 1),groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8),stride=(1, 8)),
            nn.Dropout(dropout)
        )

        # ---------- Block 3 ----------
        self.block3 = nn.Sequential(
            nn.Conv2d(16,8,kernel_size=(1, 16),dilation=(1, 2),padding="same",bias=False),
            nn.BatchNorm2d(8),
            nn.ELU(),
            nn.Conv2d(8,4,kernel_size=(1, 16),dilation=(1, 4),padding="same",bias=False),
            nn.BatchNorm2d(4),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8),stride=(1, 8)),
            nn.Dropout(dropout)
        )

        #self.Projection = nn.Sequential( Rearrange('b f i t -> b (f i) t')) # [batch,8,1,12]->[batch,8,12]

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x) # (B,4,12)
        return x

In [ ]:
class classifier_head(nn.Module):
  def __init__(self, inp_size,num_classes):
    super().__init__()
    self.classifier_layer =nn.Sequential(
        nn.Linear(inp_size,num_classes)
    )

  def forward (self, x:Tensor)->Tensor:
    x = self.classifier_layer(x)
    return x


class Projector(nn.Module):
  def __init__(self, inp=48,out=32):
    super().__init__()
    self.dropout = nn.Dropout(0.4)
    self.fc_layer =nn.Sequential(
        nn.Linear(inp, out),
        nn.GELU()
    )
  def forward (self, x:Tensor)->Tensor:
    x = self.dropout(x)
    x = self.fc_layer(x)
    return x

In [ ]:
class QSDCNN(nn.Module):

    def __init__(self,flatten_size,number_channel, number_class,n_quibit, n_qlayer, cl_size, drop_p):

        super().__init__()
        self.number_channel = number_channel
        self.n_quibit = n_quibit
        self.n_qlayer = n_qlayer
        self.cl_size = cl_size
        self.cnn = MEEGNeX(n_channels=number_channel, dropout= drop_p)
        self.projector = Projector(inp = flatten_size, out = cl_size)

        self.quantum_layers = nn.ModuleList(
                              [MyQuantumCircuit(n_qubits=self.n_quibit,n_qlayers=self.n_qlayer) for _ in range(cl_size//16)])

        self.flatten_layer = Rearrange('b m n -> b (m n)')
        self.classifier = classifier_head (cl_size//4,number_class)


    def forward(self,x):
        # x : (B,64,795)
        x = self.cnn(x)
        x = self.flatten_layer(x)
        x = self.projector(x)
        chunks = torch.chunk(x, (self.cl_size//16), dim=1)
        x_list = [layer(chunk) for layer, chunk in zip(self.quantum_layers, chunks)]
        x = torch.cat(x_list, dim=1)   # (B, 16)
        x = self.classifier(x) # x : (B,number_class)
        return x

# Utils

In [ ]:
class CustomDataset(torch.utils.data.Dataset):
  def __init__(self, eeg_data, label):
    shuffle_num = np.random.permutation(len(eeg_data))
    self.eeg_data = eeg_data[shuffle_num, :, :]  # (**, 14, 1280)
    self.label = label[shuffle_num].squeeze()

  def __len__(self):
    return len(self.eeg_data)

  def __getitem__(self, idx):
    return torch.from_numpy(self.eeg_data[idx]).float(), torch.from_numpy(np.array(self.label[idx], dtype=np.int64)).long()

In [ ]:
def numberClassChannel():
    number_class = 5
    number_channel = 64
    return number_class, number_channel


def load_data_evaluate(dir_path, n_sub, mode_evaluate="LOSO"):

    if mode_evaluate=="LOSO":
        return load_data_LOSO(dir_path, n_sub)
    else:
        return load_data_subject_dependent(dir_path, n_sub)


def load_data_subject_dependent(dir_path, n_sub):
    train_data, train_label = load_data(dir_path, n_sub, mode='train')
    test_data, test_label = load_data(dir_path,n_sub, mode='test')
    return train_data, train_label, test_data, test_label

def load_data_LOSO(dir_path, subject):
    X_train, y_train = np.empty([0, 3]), np.empty([0, 3])
    for n_sub in range (1, 16):

        X1, y1 = load_data(dir_path, n_sub, mode='train')
        X2, y2 = load_data(dir_path, n_sub, mode='test')
        X = np.concatenate((X1, X2), axis=0)
        y = np.concatenate((y1, y2), axis=0)

        if (n_sub == subject):
            X_test = X
            y_test = y
        elif X_train.shape[0] == 0:
            X_train = X
            y_train = y
        else:
            X_train = np.concatenate((X_train, X), axis=0)
            y_train = np.concatenate((y_train, y), axis=0)

    return X_train, y_train, X_test, y_test

def load_data(dir_path, n_sub, mode='train'):
    if mode=='train':
        mode_s = 'T'
    else:
        mode_s = 'V'
    data_mat = scipy.io.loadmat(dir_path + f'Sub{n_sub:02d}{mode_s}.mat')
    data = data_mat['data']  # (300, 64, 795)
    label =data_mat['label'] #(300,1)
    return data, label

def calMetrics(y_true, y_pred):

    number = max(y_true)
    if number == 2:
        mode = 'binary'
    else:
        mode = 'macro'

    accuracy = accuracy_score(y_true, y_pred)
    precison = precision_score(y_true, y_pred, average=mode)
    recall = recall_score(y_true, y_pred, average=mode)
    f1 = f1_score(y_true, y_pred, average=mode)
    kappa = cohen_kappa_score(y_true, y_pred)
    return accuracy, precison, recall, f1, kappa

# Experiment

In [ ]:
class ExP():
  def __init__(self,
               data_dir,
               nsub,
               result_name,
               epochs,
               aug_percent,
               number_seg,
               gpus,
               flatten_size,
               drop_p=0.3,
               validate_ratio=0.2,
               learning_rate=0.001,
               batch_size=300,
               sequence_length = 795,
               evaluate_mode='LOSO',
               n_quibit = 4,
               n_qlayer = 1,
               cl_size = 48):

    super(ExP, self).__init__()
    self.drop_p = drop_p
    self.qlayer = n_qlayer
    self.cl_size = cl_size
    self.quibit = n_quibit
    self.root = data_dir
    self.nSub = nsub
    self.result_name = result_name
    self.n_epochs = epochs
    self.aug_percent = aug_percent
    self.number_seg = number_seg
    self.flatten_size = flatten_size
    self.batch_size = batch_size   # each batch of raw train dataset, real training batchsize =  batch_size * (1 + N_AUG) for additional data augmentation.
    self.seq_len = sequence_length
    self.lr = learning_rate
    self.b1 = 0.5
    self.b2 = 0.999
    self.validate_ratio = validate_ratio
    self.evaluate_mode = evaluate_mode
    self.criterion_cls = torch.nn.CrossEntropyLoss().to(device )

    self.number_class, self.number_channel = numberClassChannel()
    self.model = QSDCNN(flatten_size=self.flatten_size,
                       number_channel=self.number_channel,
                       number_class=self.number_class,
                       n_quibit = self.quibit,
                       n_qlayer = self.qlayer,
                       cl_size = self.cl_size,
                       drop_p = self.drop_p)

    self.model = self.model.to(device)
    self.model_filename = self.result_name + '/model_{}.pth'.format(self.nSub)

    (self.train_data,
      self.train_label,
      self.test_data,
      self.test_label) = load_data_evaluate(self.root,self.nSub, self.evaluate_mode)

    #standardize
    target_mean = np.mean(self.train_data)
    target_std = np.std(self.train_data)
    self.train_data = (self.train_data - target_mean) / target_std
    self.test_data = (self.test_data - target_mean) / target_std

  # Segmentation and Reconstruction (S&R) data augmentation
  def interaug(self, eeg, label):
    aug_data = []
    aug_label = []
    number_records_by_augmentation = int(self.aug_percent * int(self.batch_size / self.number_class))
    number_segmentation_points = self.seq_len // self.number_seg

    for clsAug in range(self.number_class):
        cls_idx = np.where(label.squeeze() == clsAug)[0]
        tmp_data = eeg[cls_idx]
        tmp_label = label[cls_idx]

        total_tmp_labels = tmp_label.shape[0]
        if total_tmp_labels<number_records_by_augmentation:
            for i in range(number_records_by_augmentation - total_tmp_labels):
                tmp_label = np.append(tmp_label,[clsAug])


        tmp_aug_data = np.zeros((number_records_by_augmentation, self.number_channel, self.seq_len))


        for ri in range(number_records_by_augmentation):
            rand_idx = np.random.randint(0, tmp_data.shape[0], self.number_seg)
            for rj in range(self.number_seg):
                tmp_aug_data[ri, :, rj * number_segmentation_points:(rj + 1) * number_segmentation_points] = \
                    tmp_data[rand_idx[rj], :, rj * number_segmentation_points:(rj + 1) * number_segmentation_points]

        aug_data.append(tmp_aug_data)
        aug_label.append(tmp_label[:number_records_by_augmentation])



    aug_data = np.concatenate(aug_data, axis=0)
    aug_label = np.concatenate(aug_label, axis=0)


    aug_shuffle = np.random.permutation(len(aug_data))
    aug_data = aug_data[aug_shuffle]
    aug_label = aug_label[aug_shuffle]

    aug_data = torch.from_numpy(aug_data).float()
    aug_label = torch.from_numpy(aug_label.squeeze()).long()
    return aug_data, aug_label


  def train(self):

    print('-'*20, "train size：", self.train_data.shape, "test size：", self.test_data.shape)

    train_dataset = CustomDataset(self.train_data, self.train_label)
    test_dataset = CustomDataset(self.test_data, self.test_label)


    self.test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset,
                                                       batch_size=self.batch_size,
                                                       shuffle=False, pin_memory=True)

    # Optimizers
    self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, betas=(self.b1, self.b2))
    best_epoch = 0
    num = 0
    min_loss = 100

    # recording train_acc, train_loss, test_acc, test_loss for epochs
    history = []

    train_start_time = datetime.datetime.now()
    # Train the model
    for epoch in range(self.n_epochs):
        #initials
        epoch_process = {}
        epoch_process['epoch'] = epoch
        #to find the validation loss and accuracy per epoch
        val_data_list = []
        val_label_list = []

        #to find the training accuracy and loss per epoch
        total_prediction_list = []
        total_label_list = []

        self.train_dataloader = torch.utils.data.DataLoader(dataset= train_dataset,
                                                            batch_size=self.batch_size,
                                                            shuffle=True, pin_memory= True)



        self.model.train()


        for i, (batch_eeg_data, batch_label) in enumerate(self.train_dataloader):

            batch_size = batch_eeg_data.shape[0]
            number_validate = int(self.validate_ratio * batch_size)

            # split raw train dataset into real train dataset and validate dataset
            batch_train_data = batch_eeg_data[:-number_validate]
            batch_train_label = batch_label[:-number_validate]

            val_data_list.append(batch_eeg_data[-number_validate:])
            val_label_list.append(batch_label[-number_validate:])

            # data augmentation
            aug_data, aug_label = self.interaug(batch_eeg_data, batch_label) #make interaug compatible


            # concat real train dataset and generate artificial train dataset
            batch_train_data = torch.cat((batch_train_data, aug_data))
            batch_train_label = torch.cat((batch_train_label, aug_label))

            batch_train_data, batch_train_label = batch_train_data.to(device), batch_train_label.to(device)

            # training model
            batch_outputs = self.model(batch_train_data)

            total_prediction_list.append(batch_outputs)
            total_label_list.append(batch_train_label)

            loss = self.criterion_cls(batch_outputs, batch_train_label)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        del batch_train_data, batch_train_label, batch_eeg_data, batch_label, batch_outputs, aug_data, aug_label
        torch.cuda.empty_cache()

        # validation process
        if (epoch + 1) % 1 == 0:
            self.model.eval()

            # validate model
            val_data = torch.cat(val_data_list)
            val_label = torch.cat(val_label_list)

            val_dataset = torch.utils.data.TensorDataset(val_data, val_label)

            self.val_dataloader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=self.batch_size, shuffle=False)
            pred_list = []
            true_list = []
            with torch.no_grad():
                for i, (eeg,label) in enumerate(self.val_dataloader):
                    # val model
                    eeg, label = eeg.to(device), label.to(device)
                    out = self.model(eeg)
                    pred_list.append(out)
                    true_list.append(label)
                del eeg,label, out
                torch.cuda.empty_cache()

            del val_data, val_label

            pred_list = torch.cat(pred_list)
            true_list = torch.cat(true_list)

            val_loss = self.criterion_cls(pred_list, true_list)

            pred_list = torch.max(pred_list, 1)[1]
            val_acc = float((pred_list == true_list).cpu().numpy().astype(int).sum()) / float(true_list.size(0))

            del pred_list, true_list
            torch.cuda.empty_cache()

            epoch_process['val_acc'] = val_acc
            epoch_process['val_loss'] = val_loss.detach().cpu().numpy()


            total_prediction_list = torch.cat(total_prediction_list)
            total_label_list = torch.cat(total_label_list)

            total_prediction_list = torch.max(total_prediction_list, 1)[1]
            train_acc = float((total_prediction_list == total_label_list).cpu().numpy().astype(int).sum()) / float(total_label_list.size(0))

            epoch_process['train_acc'] = train_acc
            epoch_process['train_loss'] = loss.detach().cpu().numpy()

            del total_prediction_list, total_label_list
            torch.cuda.empty_cache()

            num = num + 1

            # if min_loss>val_loss:
            if val_loss < min_loss:
                min_loss = val_loss
                best_epoch = epoch
                torch.save(self.model, self.model_filename)
                print("{}_{} train_acc: {:.4f} train_loss: {:.6f}\tval_acc: {:.6f} val_loss: {:.7f}".format(self.nSub,
                                                                                        epoch_process['epoch'],
                                                                                        epoch_process['train_acc'],
                                                                                        epoch_process['train_loss'],
                                                                                        epoch_process['val_acc'],
                                                                                        epoch_process['val_loss'],))


            history.append(epoch_process)
    #end_1st_loop
    train_end_time = datetime.datetime.now()
    training_time = train_end_time-train_start_time
    # load model for test
    self.model = torch.load(self.model_filename,weights_only=False).to(device)
    self.model.eval()

    test_outputs_list = []
    test_true_labels =[]

    with torch.no_grad():
        test_start_time = datetime.datetime.now()
        for i, (eeg_test, test_label) in enumerate(self.test_dataloader):

            eeg_test,test_label = eeg_test.to(device),test_label.to(device)

            outputs = self.model(eeg_test)

            test_outputs_list.append(outputs)
            test_true_labels.append(test_label)

        test_end_time = datetime.datetime.now()
        del eeg_test, test_label, outputs
        torch.cuda.empty_cache()

    inference_time=test_end_time-test_start_time
    y_pred = torch.cat(test_outputs_list)
    y_pred = torch.max(y_pred, 1)[1]
    y_true = torch.cat(test_true_labels)



    test_acc = float((y_pred == y_true).cpu().numpy().astype(int).sum()) / float(y_true.size(0))

    print("epoch: ", best_epoch, '\tThe test accuracy is:', test_acc)


    df_history = pd.DataFrame(history)

    return test_acc, y_true, y_pred, df_history, best_epoch, training_time.total_seconds(), inference_time.total_seconds()



# Main Function

In [ ]:
def main(dirs, n_qlayer, cl_size):

    dirs
    if not os.path.exists(dirs):
        os.makedirs(dirs)

    result_write_metric = ExcelWriter(dirs+"/result_metric.xlsx")

    result_metric_dict = {}
    y_true_pred_dict = { }

    process_write = ExcelWriter(dirs+"/process_train.xlsx")
    pred_true_write = ExcelWriter(dirs+"/pred_true.xlsx")
    subjects_result = []
    best_epochs = []

    for sub in range(NUM_SUB):

        starttime = datetime.datetime.now()
        seed_n = np.random.randint(2026)
        print('seed is ' + str(seed_n))
        random.seed(seed_n)
        np.random.seed(seed_n)
        torch.manual_seed(seed_n)
        torch.cuda.manual_seed(seed_n)
        torch.cuda.manual_seed_all(seed_n)
        index_round =0
        print(f'Subject {sub+1}')
        exp = ExP(data_dir= DATA_DIR,
                  nsub= sub+1,
                  result_name=dirs,
                  epochs= EPOCHS,
                  aug_percent = AUG_PERCENT,
                  number_seg= N_SEG,
                  gpus= gpus,
                  flatten_size= FLATTEN_SIZE,
                  drop_p = DROP_P,
                  validate_ratio = VALIDATE_RATIO,
                  learning_rate = 0.001,
                  batch_size = BATCH_SIZE,
                  sequence_length= SEQUENCE_LENGTH,
                  evaluate_mode = EVALUATE_MODE,
                  n_quibit = N_QUIBIT,
                  n_qlayer = n_qlayer,
                  cl_size = cl_size
                  )


        testAcc, Y_true, Y_pred, df_process, best_epoch,training_time, inference_time = exp.train()

        true_cpu = Y_true.cpu().numpy().astype(int)
        pred_cpu = Y_pred.cpu().numpy().astype(int)
        df_pred_true = pd.DataFrame({'pred': pred_cpu, 'true': true_cpu})
        df_pred_true.to_excel(pred_true_write, sheet_name=str(sub))
        y_true_pred_dict[sub] = df_pred_true

        accuracy, precison, recall, f1, kappa = calMetrics(true_cpu, pred_cpu)
        subject_result = {'accuray': accuracy*100,
                          'precision': precison*100,
                          'recall': recall*100,
                          'f1': f1*100,
                          'kappa': kappa*100,
                          'training_time': training_time,
                          'inference_time': inference_time
                          }
        subjects_result.append(subject_result)

        df_process.to_excel(process_write, sheet_name=str(sub))
        best_epochs.append(best_epoch)

        print(' THE BEST ACCURACY IS ' + str(testAcc) + "\tkappa is " + str(kappa) )

        endtime = datetime.datetime.now()
        print('subject %d duration: '%(sub+1) + str(endtime - starttime))

    df_result = pd.DataFrame(subjects_result)
    process_write.close()

    pred_true_write.close()

    print('**The average Best accuracy is: ' + str(df_result['accuray'].mean()) + "kappa is: " + str(df_result['kappa'].mean()) + "\n" )
    print("best epochs: ", best_epochs)
    df_result.to_excel(result_write_metric, index=False)
    result_metric_dict = df_result

    mean = df_result.mean(axis=0)
    mean.name = 'mean'
    std = df_result.std(axis=0)
    std.name = 'std'
    df_result = pd.concat([df_result, pd.DataFrame(mean).T, pd.DataFrame(std).T])

    df_result.to_excel(result_write_metric, index=False)

    print('-'*9, ' all result ', '-'*9)
    print(df_result)

    print("*"*40)

    result_write_metric.close()


    return result_metric_dict

# Control Section

In [ ]:
if __name__ == "__main__":
    #----------------------------------------
    print("*"*65)
    print(" "*18+"Working in: ", os.getcwd())
    print("*"*65)
    print("*"*65)


    EVALUATE_MODE= 'Subdep'
    VALIDATE_RATIO = 0.3
    FLATTEN_SIZE = 48
    NUM_SUB= 15
    N_QLAYER = [1,2]
    CL_SIZE = [16,32,48]
    N_QUIBIT = 4
    SEQUENCE_LENGTH = 795
    DATA_DIR = "G:\Imagined Speech\BCID Dataset/"


    #Hyperparameters For Experimental Setup
    AUG_PERCENT = 1.0   # how much augmented data
    N_SEG = 5           # segments times in S&R operaion
    EPOCHS = 1000
    BATCH_SIZE = 300

    if EVALUATE_MODE == 'LOSO':
      DROP_P = 0.2        # dropout rates in CNN
    else:
      DROP_P = 0.4

    for k in range (3):
      for cl_size in CL_SIZE:
        for n_qlayer in N_QLAYER:
          RESULT_NAME = f"{EVALUATE_MODE}_BCI2020T3_Branch_{cl_size//16}_Layer_{n_qlayer}_{k}"
          print("="*65)
          print(" "*14+f"{EVALUATE_MODE}_BCI2020T3_Branch_{cl_size//16}_Layer_{n_qlayer}_{k}")
          print("*"*65)
          print("="*65)
          print(time.asctime(time.localtime(time.time())))
          result = main(RESULT_NAME, n_qlayer, cl_size )
          print(time.asctime(time.localtime(time.time())))